# Lab 03 — Data Integration dan Data Transformation
**Multi-format, schema, quality, JDBC, join/union, lineage, curated dataset**

Notebook ini dirancang untuk environment pelatihan: Spark 3.5.7 standalone, Jupyter, 2 worker, masing-masing 2 core / 2 GB.  
**Prinsip lab:** prediksi → jalankan → amati → jelaskan. Jangan hanya mengejar output akhir.

## Capaian praktik
Pola utama: **INGEST → INSPECT → TRANSFORM → INTEGRATE → VALIDATE → PUBLISH**. Peserta mengolah CSV, JSON, Parquet/ORC, JDBC, join/union, lineage, dan quality checks.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
import time, json, os, shutil

spark = (
    SparkSession.builder
    .appName("spark-training-lab")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("App name      :", spark.sparkContext.appName)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

## Persiapan dataset sintetis
Cell ini membuat file latihan di `/home/spark/datasets/integration_lab`. Tidak memakai data produksi.

In [ ]:
base_dir = "/home/spark/datasets/integration_lab"
os.makedirs(base_dir, exist_ok=True)

legacy_rows = [
    ("L001", " 10001 ", " jakarta ", "2026-10-01", "1250000.50", "VALID"),
    ("L002", "10002", "Bandung", "2026-10-02", "250000", "VALID"),
    ("L003", None, "Surabaya", "2026-10-03", "900000", "VALID"),
    ("L004", "10003", "JAKARTA", "bad-date", "-100", "VALID"),
    ("L004", "10003", "JAKARTA", "bad-date", "-100", "VALID"),
]
legacy_schema = "transaction_id string, taxpayer_id string, city string, event_date string, amount string, status string"
legacy_seed = spark.createDataFrame(legacy_rows, legacy_schema)
legacy_path = f"{base_dir}/transactions_legacy.csv"
legacy_seed.coalesce(1).write.mode("overwrite").option("header", True).csv(legacy_path)

api_json = [
    '{"id":"A101","taxpayer":{"id":"10001","city":"Jakarta"},"event":{"date":"2026-10-04"},"amount":5000000,"status":"VALID"}',
    '{"id":"A102","taxpayer":{"id":"10004","city":"Bogor"},"event":{"date":"2026-10-05"},"amount":750000,"status":"VALID"}',
    '{"id":"A103","taxpayer":{"id":"99999","city":"Depok"},"event":{"date":"2026-10-05"},"amount":120000,"status":"VALID"}',
]
api_path = f"{base_dir}/transactions_api.json"
spark.createDataFrame([(x,) for x in api_json], ["value"]).coalesce(1).write.mode("overwrite").text(api_path)

print("CSV :", legacy_path)
print("JSON:", api_path)

## Lab 1 — Normalize dirty DataFrame (30 menit)
**Tujuan:** membuat DataFrame typed, bersih, dan deduplicated; setiap record yang dibuang harus dapat dijelaskan.

In [ ]:
raw_legacy = spark.read.option("header", True).csv(legacy_path)
raw_legacy.show(truncate=False)
raw_legacy.printSchema()

In [ ]:
profile = raw_legacy.select(
    F.count("*").alias("rows"),
    F.countDistinct("transaction_id").alias("distinct_trx"),
    F.sum(F.col("taxpayer_id").isNull().cast("int")).alias("null_taxpayer"),
)
profile.show()

**Tugas:** trim ID/city, uppercase city, cast amount ke decimal, parse date, buang mandatory key null, definisikan kebijakan duplicate, pisahkan record invalid (tanggal gagal parse / amount negatif).

In [ ]:
legacy_clean = (
    raw_legacy
    .withColumn("taxpayer_id", F.trim("taxpayer_id"))
    .withColumn("city", F.upper(F.trim("city")))
    .withColumn("event_date", F.to_date("event_date", "yyyy-MM-dd"))
    .withColumn("amount", F.col("amount").cast("decimal(18,2)"))
    .dropDuplicates(["transaction_id"])
)

invalid_legacy = legacy_clean.filter(
    F.col("taxpayer_id").isNull() |
    F.col("event_date").isNull() |
    (F.col("amount") < 0)
)
valid_legacy = legacy_clean.filter(
    F.col("taxpayer_id").isNotNull() &
    F.col("event_date").isNotNull() &
    (F.col("amount") >= 0)
)

print("raw    :", raw_legacy.count())
print("valid  :", valid_legacy.count())
print("invalid:", invalid_legacy.count())
invalid_legacy.show(truncate=False)

## Lab 2 — Multi-format ingestion + canonical schema (35 menit)
Baca JSON nested, flatten, samakan schema dengan legacy, `unionByName()`, tambah lineage, lalu tulis Parquet terpartisi.

In [ ]:
api_schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("taxpayer", T.StructType([
        T.StructField("id", T.StringType()),
        T.StructField("city", T.StringType()),
    ])),
    T.StructField("event", T.StructType([
        T.StructField("date", T.StringType()),
    ])),
    T.StructField("amount", T.DecimalType(18,2)),
    T.StructField("status", T.StringType()),
])

api = spark.read.schema(api_schema).json(api_path)
api.printSchema()
api.show(truncate=False)

In [ ]:
legacy_std = (
    valid_legacy
    .select(
        "transaction_id", "taxpayer_id", "city", "event_date", "amount", "status"
    )
    .withColumn("_source", F.lit("legacy_csv"))
)

api_std = (
    api.select(
        F.col("id").alias("transaction_id"),
        F.col("taxpayer.id").alias("taxpayer_id"),
        F.upper(F.trim(F.col("taxpayer.city"))).alias("city"),
        F.to_date(F.col("event.date"), "yyyy-MM-dd").alias("event_date"),
        F.col("amount").cast("decimal(18,2)").alias("amount"),
        "status",
    )
    .withColumn("_source", F.lit("api_json"))
)

all_trx = (
    legacy_std.unionByName(api_std)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("year", F.year("event_date"))
    .withColumn("month", F.month("event_date"))
)
all_trx.show(truncate=False)

In [ ]:
curated_stage = f"{base_dir}/curated_stage"
(all_trx.write
 .mode("overwrite")
 .partitionBy("year", "month")
 .parquet(curated_stage))

roundtrip = spark.read.parquet(curated_stage)
print("roundtrip rows:", roundtrip.count())
roundtrip.printSchema()

**Opsional ORC:** tulis DataFrame yang sama sebagai ORC lalu bandingkan schema saat dibaca kembali.

In [ ]:
orc_path = f"{base_dir}/curated_orc"
all_trx.write.mode("overwrite").orc(orc_path)
spark.read.orc(orc_path).show(3, truncate=False)

## Lab 3 — JDBC PostgreSQL reference data (40 menit)
Environment compose menyediakan PostgreSQL di `postgres:5432`, database `training`, user/password `spark`. Spark tetap membutuhkan PostgreSQL JDBC driver pada driver dan executor.

Cell berikut melakukan **preflight**. Jika driver belum tersedia, lanjutkan ke fallback DataFrame agar konsep join tetap dapat dipraktikkan.

In [ ]:
try:
    spark._jvm.java.lang.Class.forName("org.postgresql.Driver")
    jdbc_available = True
except Exception as e:
    jdbc_available = False
    print("PostgreSQL JDBC driver belum tersedia di Spark classpath.")
    print("Trainer: pre-bundle postgresql JDBC JAR sebelum kelas.")
print("jdbc_available =", jdbc_available)

In [ ]:
jdbc_url = "jdbc:postgresql://postgres:5432/training"
props = {"user": "spark", "password": "spark", "driver": "org.postgresql.Driver"}

if jdbc_available:
    try:
        taxpayer_master = spark.read.jdbc(jdbc_url, "public.taxpayer_master", properties=props)
        taxpayer_master.show(5, truncate=False)
    except Exception as e:
        print("Table public.taxpayer_master belum tersedia. Menggunakan fallback synthetic master.")
        jdbc_available = False

if not jdbc_available:
    taxpayer_master = spark.createDataFrame([
        ("10001", "SEG_A", "JKT01"),
        ("10002", "SEG_B", "BDG01"),
        ("10003", "SEG_A", "JKT02"),
        ("10004", "SEG_C", "BGR01"),
    ], ["taxpayer_id", "segment", "office_code"])

taxpayer_master.show(truncate=False)

## Lab 4 — Join cardinality, match rate, quarantine (30 menit)
Sebelum join, cek apakah `taxpayer_id` unik di master. Lalu lakukan left join dan hitung matched/unmatched.

In [ ]:
master_dupes = (taxpayer_master.groupBy("taxpayer_id").count().filter(F.col("count") > 1))
print("duplicate keys in master:", master_dupes.count())
master_dupes.show()

m = taxpayer_master.select("taxpayer_id", "segment", "office_code").alias("m")
t = all_trx.alias("t")

enriched = (
    t.join(m, F.col("t.taxpayer_id") == F.col("m.taxpayer_id"), "left")
     .select("t.*", F.col("m.segment"), F.col("m.office_code"))
)

matched = enriched.filter(F.col("segment").isNotNull())
unmatched = enriched.filter(F.col("segment").isNull())
print("all      :", enriched.count())
print("matched  :", matched.count())
print("unmatched:", unmatched.count())
unmatched.show(truncate=False)

## Lab 5 — Capstone curated dataset (45–55 menit)
**Deliverable peserta:**
1. Ingest CSV + JSON dengan lineage.
2. Standardisasi schema/types.
3. Union ke canonical schema.
4. Enrich master data.
5. Buat minimal 3 quality metrics.
6. Pisahkan unmatched ke quarantine.
7. Tulis curated Parquet dengan partition column yang masuk akal.
8. Read-back dan validasi count/schema.

Gunakan function agar notebook mudah diubah menjadi transformation-as-code.

In [ ]:
def quality_report(df):
    return df.select(
        F.count("*").alias("rows"),
        F.sum(F.col("taxpayer_id").isNull().cast("int")).alias("null_taxpayer"),
        F.sum(F.col("amount").isNull().cast("int")).alias("null_amount"),
        F.sum((F.col("amount") < 0).cast("int")).alias("negative_amount"),
        F.countDistinct("transaction_id").alias("distinct_transactions"),
    )

quality_report(enriched).show(truncate=False)

final_curated = enriched.filter(F.col("segment").isNotNull())
final_path = f"{base_dir}/final_curated"
quarantine_path = f"{base_dir}/quarantine_unmatched"

(final_curated
 .withColumn("report_month", F.date_format("event_date", "yyyy-MM"))
 .write.mode("overwrite")
 .partitionBy("report_month")
 .parquet(final_path))

unmatched.write.mode("overwrite").parquet(quarantine_path)

print("final rows:", spark.read.parquet(final_path).count())
print("quarantine rows:", spark.read.parquet(quarantine_path).count())

## Exit ticket
Mengapa keberhasilan `write()` belum cukup untuk menyatakan transformasi selesai? Sebutkan minimal 5 validasi yang harus dilakukan sebelum publish.